##**1. 문자 단위 RNN**

: RNN의 입출력의 단위가 단어 레벨이 아니라 문자 레벨로 하여 RNN을 구현한다면, 이를 문자 단위 RNN이라고 함.

- 구조 자체가 달라진 것은 아니고, 입, 출력의 단위가 문자로 바뀌었을 뿐.


In [1]:
# 필요 도구 임포트
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

###**1) 훈련 데이터 전처리**



In [2]:
# 입력 데이터와 레이블 데이터에 대해 문자 집합 만들기
# 문자 집합이란 중복을 제거한 문자들의 집합
input_str = 'apple'
label_str = 'pple!'
char_vocab = sorted(list(set(input_str+label_str)))
vocab_size = len(char_vocab)
print ('문자 집합의 크기 : {}'.format(vocab_size))

문자 집합의 크기 : 5


- 현재 문자 집합에는 총 5개의 문자 존재.
> !, a, e, l, p

In [3]:
# 하이퍼파라미터 정의
# 입력은 원-핫 벡터 사용할 것이기에 입력의 크기는 문자 집합의 크기여야만 함
input_size = vocab_size # 입력의 크기는 문자 집합의 크기
hidden_size = 5
output_size = 5
learning_rate = 0.1

In [4]:
# 문자 집합에 고유한 정수 부여
char_to_index = dict((c, i) for i, c in enumerate(char_vocab)) # 문자에 고유한 정수 인덱스 부여
print(char_to_index)

{'!': 0, 'a': 1, 'e': 2, 'l': 3, 'p': 4}


- !은 0, a는 1, e는 2, l은 3, p는 4가 부여

In [5]:
# 나중에 예측 결과를 다시 문자 시퀀스로 보기 위해 반대로 정수로부터 문자를 얻을 수 있는 index_to_char 생성
index_to_char={}
for key, value in char_to_index.items():
    index_to_char[value] = key
print(index_to_char)

{0: '!', 1: 'a', 2: 'e', 3: 'l', 4: 'p'}


In [6]:
# 입력 데이터와 레이블 데이터의 각 문자들을 정수로 맵핑
x_data = [char_to_index[c] for c in input_str]
y_data = [char_to_index[c] for c in label_str]
print(x_data)
print(y_data)

[1, 4, 4, 3, 2]
[4, 4, 3, 2, 0]


- 파이토치의 nn.RNN()은 기본적으로 3차원 텐서를 입력받음. 따라서 배치 차원 추가.

In [7]:
# 배치 차원 추가
# 텐서 연산인 unsqueeze(0)를 통해 해결할 수도 있었음.
x_data = [x_data]
y_data = [y_data]
print(x_data)
print(y_data)

[[1, 4, 4, 3, 2]]
[[4, 4, 3, 2, 0]]


In [8]:
# 입력 시퀀스의 각 문자들을 원-핫 벡터로 바꾸기
x_one_hot = [np.eye(vocab_size)[x] for x in x_data]
print(x_one_hot)

[array([[0., 1., 0., 0., 0.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 0., 1.],
       [0., 0., 0., 1., 0.],
       [0., 0., 1., 0., 0.]])]


In [9]:
# 입력 데이터와 레이블 데이터를 텐서로 바꾸기
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)

/tmp/ipykernel_3196/2102937852.py:2: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:252.)
  X = torch.FloatTensor(x_one_hot)


In [10]:
# 텐서 크기 확인
print('훈련 데이터의 크기 : {}'.format(X.shape))
print('레이블의 크기 : {}'.format(Y.shape))

훈련 데이터의 크기 : torch.Size([1, 5, 5])
레이블의 크기 : torch.Size([1, 5])


###**2) 모델 구현하기**

- 아래에서 fc는 완전 연결층을 의미하며 출력층으로 사용됨


In [11]:
class Net(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_size, hidden_size, batch_first=True) # RNN 셀 구현
        self.fc = torch.nn.Linear(hidden_size, output_size, bias=True) # 출력층 구현

    def forward(self, x): # 구현한 RNN 셀과 출력층을 연결
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x

In [12]:
# 클래스로 정의한 모델을 net에 저장
net = Net(input_size, hidden_size, output_size)

In [13]:
# 입력된 모델에 입력을 넣어 출력의 크기 확인
outputs = net(X)
print(outputs.shape) # 3차원 텐서

torch.Size([1, 5, 5])


In [14]:
# view를 사용하여 배치 차원과 시점 차원을 하나로 병합. (정확도 측정 시 쓰임)
print(outputs.view(-1, input_size).shape) # 2차원 텐서로 변환

torch.Size([5, 5])


In [15]:
# 레이블 데이터의 크기 확인
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([1, 5])
torch.Size([5])


In [16]:
# 옵티마이저와 손실 함수 정의
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

In [17]:
# 총 100번의 에포크 학습
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X)
    loss = criterion(outputs.view(-1, input_size), Y.view(-1)) # view를 하는 이유는 Batch 차원 제거를 위해
    loss.backward() # 기울기 계산
    optimizer.step() # 아까 optimizer 선언 시 넣어둔 파라미터 업데이트

    # 아래 세 줄은 모델이 실제 어떻게 예측했는지를 확인하기 위한 코드.
    result = outputs.data.numpy().argmax(axis=2) # 최종 예측값인 각 time-step 별 5차원 벡터에 대해서 가장 높은 값의 인덱스를 선택
    result_str = ''.join([index_to_char[c] for c in np.squeeze(result)])
    print(i, "loss: ", loss.item(), "prediction: ", result, "true Y: ", y_data, "prediction str: ", result_str)

0 loss:  1.5434353351593018 prediction:  [[2 4 2 2 4]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  epeep
1 loss:  1.3243591785430908 prediction:  [[4 4 4 2 4]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppep
2 loss:  1.1280758380889893 prediction:  [[4 4 4 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppe!
3 loss:  0.932951807975769 prediction:  [[4 4 4 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppe!
4 loss:  0.7479735612869263 prediction:  [[4 4 4 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppe!
5 loss:  0.5725804567337036 prediction:  [[4 4 4 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pppe!
6 loss:  0.43220824003219604 prediction:  [[4 4 3 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pple!
7 loss:  0.3250831663608551 prediction:  [[4 4 3 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pple!
8 loss:  0.23707565665245056 prediction:  [[4 4 3 2 0]] true Y:  [[4, 4, 3, 2, 0]] prediction str:  pple!
9 loss:  0.16410502791404724 prediction:  [[4 4 3 2 0]

##**2. 더 많은 데이터로 학습한 문자 단위 RNN**

In [18]:
# 필요 도구 임포트
import torch
import torch.nn as nn
import torch.optim as optim

###**1) 훈련 데이터 전처리하기**

In [19]:
sentence = ("if you want to build a ship, don't drum up people together to "
            "collect wood and don't assign them tasks and work, but rather "
            "teach them to long for the endless immensity of the sea.")

In [20]:
# 문자 집합 생성 후 각 문자에 고유한 정수 부여
char_set = list(set(sentence)) # 중복을 제거한 문자 집합 생성
char_dic = {c: i for i, c in enumerate(char_set)} # 각 문자에 정수 인코딩

In [21]:
print(char_dic) # 공백도 여기서는 하나의 원소

{'u': 0, 'd': 1, 'i': 2, 'e': 3, 'y': 4, "'": 5, 'o': 6, 'h': 7, 'f': 8, 'k': 9, 'b': 10, 't': 11, 'g': 12, ',': 13, 'l': 14, ' ': 15, 'n': 16, 'm': 17, '.': 18, 'w': 19, 's': 20, 'p': 21, 'c': 22, 'a': 23, 'r': 24}


- 각 문자에 정수 부여되었으며, 총 25개의 문자 존재

In [22]:
# 문자 집합의 크기 확인
dic_size = len(char_dic)
print('문자 집합의 크기 : {}'.format(dic_size))

문자 집합의 크기 : 25


In [23]:
# 하이퍼파라미터 설정
hidden_size = dic_size
sequence_length = 10  # 임의 숫자 지정
learning_rate = 0.1

- hidden_size(은닉 상태의 크기)를 입력의 크기와 동일하게 줌
- sequence_length라는 변수는 앞서 만든 샘플을 10개 단위로 끊어서 샘플을 만들 예정이기 때문

In [24]:
# sequence_length 값인 10의 단위로 샘플들을 잘라 데이터를 만들기
# 데이터 구성
x_data = []
y_data = []

for i in range(0, len(sentence) - sequence_length):
    x_str = sentence[i:i + sequence_length]
    y_str = sentence[i + 1: i + sequence_length + 1]
    print(i, x_str, '->', y_str)

    x_data.append([char_dic[c] for c in x_str])  # x str to index
    y_data.append([char_dic[c] for c in y_str])  # y str to index

0 if you wan -> f you want
1 f you want ->  you want 
2  you want  -> you want t
3 you want t -> ou want to
4 ou want to -> u want to 
5 u want to  ->  want to b
6  want to b -> want to bu
7 want to bu -> ant to bui
8 ant to bui -> nt to buil
9 nt to buil -> t to build
10 t to build ->  to build 
11  to build  -> to build a
12 to build a -> o build a 
13 o build a  ->  build a s
14  build a s -> build a sh
15 build a sh -> uild a shi
16 uild a shi -> ild a ship
17 ild a ship -> ld a ship,
18 ld a ship, -> d a ship, 
19 d a ship,  ->  a ship, d
20  a ship, d -> a ship, do
21 a ship, do ->  ship, don
22  ship, don -> ship, don'
23 ship, don' -> hip, don't
24 hip, don't -> ip, don't 
25 ip, don't  -> p, don't d
26 p, don't d -> , don't dr
27 , don't dr ->  don't dru
28  don't dru -> don't drum
29 don't drum -> on't drum 
30 on't drum  -> n't drum u
31 n't drum u -> 't drum up
32 't drum up -> t drum up 
33 t drum up  ->  drum up p
34  drum up p -> drum up pe
35 drum up pe -> rum up peo
36

In [25]:
# 첫번째 샘플의 입력 데이터와 레이블 데이터 출력
print(x_data[0])
print(y_data[0])

[2, 8, 15, 4, 6, 0, 15, 19, 23, 16]
[8, 15, 4, 6, 0, 15, 19, 23, 16, 11]


- 한 칸씩 쉬프트 된 시퀀스가 정상적으로 출력되는 것을 확인할 수 있음

In [26]:
# 입력 시퀀스에 대해 원-핫 인코딩을 수행하고, 입력 데이터와 레이블 데이터를 텐서로 변환
x_one_hot = [np.eye(dic_size)[x] for x in x_data] # x 데이터는 원-핫 인코딩
X = torch.FloatTensor(x_one_hot)
Y = torch.LongTensor(y_data)

In [27]:
# 훈련 데이터와 레이블 데이터의 크기 확인
print('훈련 데이터의 크기 : {}'.format(X.shape))
print('레이블의 크기 : {}'.format(Y.shape))

훈련 데이터의 크기 : torch.Size([170, 10, 25])
레이블의 크기 : torch.Size([170, 10])


In [28]:
# 첫번째 샘플만 출력하여 확인
print(X[0])

tensor([[0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0.,
         0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 1., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,

In [29]:
# 레이블 데이터의 첫번째 샘플 출력
print(Y[0])

tensor([ 8, 15,  4,  6,  0, 15, 19, 23, 16, 11])


###**2) 모델 구현하기**


In [30]:
# 은닉층 두개 구현
class Net(torch.nn.Module):
    def __init__(self, input_dim, hidden_dim, layers): # 현재 hidden_size는 dic_size와 같음.
        super(Net, self).__init__()
        self.rnn = torch.nn.RNN(input_dim, hidden_dim, num_layers=layers, batch_first=True)
        self.fc = torch.nn.Linear(hidden_dim, hidden_dim, bias=True)

    def forward(self, x):
        x, _status = self.rnn(x)
        x = self.fc(x)
        return x

In [33]:
net = Net(dic_size, hidden_size, 2) # 이번에는 층을 두 개 쌓습니다.

- nn.RNN() 안에 num_layers라는 인자를 사용하여 은닉층을 몇 개 쌓을 건지 설정할 수 있음

In [34]:
# 비용함수와 옵티마이저 선언
criterion = torch.nn.CrossEntropyLoss()
optimizer = optim.Adam(net.parameters(), learning_rate)

In [35]:
# 모델에 입력을 넣어 출력 크기 확인
outputs = net(X)
print(outputs.shape) # 3차원 텐서

torch.Size([170, 10, 25])


In [36]:
# view 사용하여 배치 차원과 시점 차원 하나로 만들기
print(outputs.view(-1, dic_size).shape) # 2차원 텐서로 변환.

torch.Size([1700, 25])


In [37]:
# 레이블 데이터의 크기 확인
print(Y.shape)
print(Y.view(-1).shape)

torch.Size([170, 10])
torch.Size([1700])


In [38]:
# 옵티마이저와 손실 함수 정의
for i in range(100):
    optimizer.zero_grad()
    outputs = net(X) # (170, 10, 25) 크기를 가진 텐서를 매 에포크마다 모델의 입력으로 사용
    loss = criterion(outputs.view(-1, dic_size), Y.view(-1))
    loss.backward()
    optimizer.step()

    # results의 텐서 크기는 (170, 10)
    results = outputs.argmax(dim=2)
    predict_str = ""
    for j, result in enumerate(results):
        if j == 0: # 처음에는 예측 결과를 전부 가져오지만
            predict_str += ''.join([char_set[t] for t in result])
        else: # 그 다음에는 마지막 글자만 반복 추가
            predict_str += char_set[result[-1]]

    print(predict_str)

rrrprpnpprpnprpprpnrpprprprrprprrpnrrprpprprprprrnprprpnprpprpprrrpppnprrrpprrprpprpnprrpprnprprnprprprprrpprrprnprpnrpprprpprpprpprprpprprrprprrrpprpnprrrrppnprrprppprpprrprpnprp
t tr  tr   to tp  d o t t d t o t drt  t    o do t ort do t o t trrt tr  rt o tro t tr    rt to t r   tro tr  o d r drr to d r t t eo d r r rttro t erttr dr  t   t     o ot tr tr 
                                        h      o     t  h     t e     h     e                                    h   h   o         e              t  h                  a          
ty.oom.memeeye emem tyememe emememememera eoceiemtyemememe ermt eme eem mece  ememememe ece ewmee.ct eece emd ememe egd eme eceeeememeeereonemememtmememecememece eme eietet eoemec
tt toetoetheoetht t t toeoloeoooo t eo thet theo theoeoeo theoh eo ohtoet t t toeot eoeoao t eo toetheo t to to tht toeoeo toeoheaheo toethetheohetheoeoo theoeo  toeoheoheo theoeo
 t t t a ttt t a it t a d a t a t t a ieo ea tlan  a a d a ao t doat t m a a  t a t t t t t aoot a d